In [16]:
!pip install lxml requests pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 143.3 MB/s  0:00:00148.8 MB/s eta 0:00:01

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
from ib_insync import *
import pandas as pd
import time

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 4002, clientId=1)
print("Connecté ✅")

Connecté ✅


Error 1100, reqId -1: La connexion entre IBKR et Trader Workstation a \u00e9t\u00e9 perdue.
Error 1102, reqId -1: La connexion entre IBKR et Trader Workstation a \u00e9t\u00e9 restaur\u00e9e et les donn\u00e9es ont \u00e9t\u00e9 r\u00e9cup\u00e9r\u00e9es. Toutes les data farms sont connect\u00e9es: usopt; usfarm; ushmds; secdefil.
Error 1100, reqId -1: La connexion entre IBKR et Trader Workstation a \u00e9t\u00e9 perdue.
Error 1102, reqId -1: La connexion entre IBKR et Trader Workstation a \u00e9t\u00e9 restaur\u00e9e et les donn\u00e9es ont \u00e9t\u00e9 r\u00e9cup\u00e9r\u00e9es. Toutes les data farms sont connect\u00e9es: usopt; usfarm; ushmds; secdefil.
Error 1100, reqId -1: La connexion entre IBKR et Trader Workstation a \u00e9t\u00e9 perdue.
Error 1102, reqId -1: La connexion entre IBKR et Trader Workstation a \u00e9t\u00e9 restaur\u00e9e et les donn\u00e9es ont \u00e9t\u00e9 r\u00e9cup\u00e9r\u00e9es. Toutes les data farms sont connect\u00e9es: usopt; usfarm; ushmds; secdefil.
E

In [10]:
import requests
import io

headers = {'User-Agent': 'Mozilla/5.0'}
r = requests.get('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', headers=headers)
table = pd.read_html(io.StringIO(r.text))
sp500 = table[0]
tickers = sorted(sp500['Symbol'].tolist())
tickers = [t.replace('.', ' ') for t in tickers]
print(f"{len(tickers)} tickers récupérés")
print(tickers[:15])

503 tickers récupérés
['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES']


In [12]:
resolved = []
failed = []

for i in range(0, len(tickers), 50):
    batch = tickers[i:i+50]
    contracts = [Stock(t, 'SMART', 'USD') for t in batch]
    ib.qualifyContracts(*contracts)
    
    for contract in contracts:
        if contract.conId > 0:
            resolved.append(contract)
        else:
            failed.append(contract.symbol)
    
    print(f"Batch {i//50 + 1}/{len(tickers)//50 + 1}: {len([c for c in contracts if c.conId > 0])}/{ len(batch)} résolus")
    time.sleep(1)

print(f"\n✅ {len(resolved)} contrats résolus")
if failed:
    print(f"❌ {len(failed)} échecs: {failed}")

Batch 1/11: 50/50 résolus
Batch 2/11: 50/50 résolus
Batch 3/11: 50/50 résolus
Batch 4/11: 50/50 résolus
Batch 5/11: 50/50 résolus
Batch 6/11: 50/50 résolus
Batch 7/11: 50/50 résolus
Batch 8/11: 50/50 résolus
Batch 9/11: 50/50 résolus
Batch 10/11: 50/50 résolus
Batch 11/11: 3/3 résolus

✅ 503 contrats résolus


In [30]:
master = pd.DataFrame([{
    'symbol' : c.symbol,
    'conId' : c.conId,
    'exchange' : c.exchange,
    'currency' : c.currency,
    'primaryExchange' : c.primaryExchange,

} for c in resolved])

master.to_parquet('../data/parquet/instrument_master_sp500.parquet', index=False, engine='fastparquet')
print(f"✅ Instrument master sauvegardé : {len(master)} actions")
master.head(10)

✅ Instrument master sauvegardé : 503 actions


,symbol,conId,exchange,currency,primaryExchange
0,A,1715006,SMART,USD,NYSE
1,AAPL,265598,SMART,USD,NASDAQ
2,ABBV,118089500,SMART,USD,NYSE
3,ABNB,459530964,SMART,USD,NASDAQ
4,ABT,4065,SMART,USD,NYSE
5,ACGL,10763362,SMART,USD,NASDAQ
6,ACN,67889930,SMART,USD,NYSE
7,ADBE,265768,SMART,USD,NASDAQ
8,ADI,4157,SMART,USD,NASDAQ
9,ADM,4165,SMART,USD,NYSE
